# Schedule Email Sender — Expedia VN

In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — CONFIGURATION  ← Edit here each week
# ═══════════════════════════════════════════════════════════════
WEEK = "2026_08_31"
FOLDER_LINK = "https://cnxmail.sharepoint.com/:f:/r/sites/WFM-Expedia-HCM/Branding%20files/Schedule/Schedule%20(Ops%20version)/2026/03-Mar?d=w0c2afc8722a34dda9df94fa4210d6195&csf=1&web=1&e=egPcdi"
WO_SWAP_LINK = "https://cnxmail.sharepoint.com/:x:/s/WFM-Expedia-HCM/IQAGptitR5FiTZNPELbLlpObAa0GyjP_bu_NXF1vMqQBp68?e=u8bq4R"
SEND_EMAIL = True
SEND_FINAL = True
EMAIL_TO = (
    "van.tran@concentrix.com;"
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "myduyen.ly@concentrix.com;"
    "mylinh.do@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
)

EMAIL_CC = (
    "urmila.chakka1@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "aas.mohammad@concentrix.com;"
    "ASEAN_Planning@concentrix.com;"
)

In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — ALL LOGIC
# ═══════════════════════════════════════════════════════════════
import os, re, time, subprocess, pythoncom, psutil, win32com.client
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display, HTML

HOME = os.path.expanduser("~").replace("\\", "/")

SCHEDULE_OPS_FOLDER = (
    f"{HOME}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
    f"/Schedule/Schedule (Ops version)/2026"
)

# ── Validate config ──────────────────────────────────────────
if not FOLDER_LINK.strip():
    raise ValueError("FOLDER_LINK is empty! Go to SharePoint → open the month folder → Share → Copy link.")
if not FOLDER_LINK.strip().startswith("http"):
    raise ValueError(f"FOLDER_LINK must start with https://  Got: {FOLDER_LINK!r}")

print("[Config]")
print(f"  Schedule folder : {'OK' if Path(SCHEDULE_OPS_FOLDER).exists() else 'MISSING'}")
print(f"  Folder link     : {FOLDER_LINK[:70]}...")
print(f"  WO Swap link    : {WO_SWAP_LINK[:70]}...")

# ── File finder ──────────────────────────────────────────────
def get_latest_schedule_file(folder: str) -> Path:
    """Scan recursively for Schedule_WB*.xlsx, return file with highest MMDD."""
    files = [
        f for f in Path(folder).rglob("*Schedule_WB*.xlsx")
        if not f.name.startswith("~$")
    ]
    if not files:
        raise FileNotFoundError(f"No Schedule_WB*.xlsx found in: {folder}")
    def _wb_key(p: Path) -> int:
        m = re.search(r"WB(\d{4})", p.name)
        return int(m.group(1)) if m else 0
    latest = max(files, key=_wb_key)
    print(f"\n[File] {latest.name}")
    print(f"       {latest}")
    return latest

# ── Week auto-detect ─────────────────────────────────────────
def auto_detect_week(file_path: Path, folder: str) -> str:
    """Extract YYYY_MM_DD from WB<MM><DD> filename + folder year."""
    m = re.search(r"WB(\d{2})(\d{2})", file_path.name)
    if not m:
        raise ValueError(
            f"Cannot parse week from '{file_path.name}'. "
            f"Expected WB<MM><DD> e.g. WB0831. Set WEEK manually in Cell 1."
        )
    mm, dd   = m.group(1), m.group(2)
    year_str = Path(folder.rstrip("/")).name
    week_str = f"{year_str}_{mm}_{dd}"
    print(f"[Week] Auto-detected: {week_str}")
    return week_str

# ── Week label parser ────────────────────────────────────────
def parse_week_label(week_str: str) -> tuple:
    clean  = re.sub(r"^Schedule_", "", week_str.strip())
    parts  = clean.split("_")
    if len(parts) < 3:
        raise ValueError(f"Cannot parse week from '{week_str}'. Expected YYYY_MM_DD.")
    monday = datetime(int(parts[0]), int(parts[1]), int(parts[2]))
    sunday = monday + timedelta(days=6)
    return monday.strftime("%d/%m/%Y"), sunday.strftime("%d/%m/%Y")

# ── HTML email body ──────────────────────────────────────────
def build_html_body(
    file_stem: str, folder_link: str,
    wo_swap_link: str, monday_lbl: str, sunday_lbl: str
) -> str:
    FONT = "font-family:Calibri,Arial,sans-serif;"
    file_xlsx = f"{file_stem}.xlsx"
    return f"""
    <!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>
    <o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->
    <div style="padding:24px 28px;background:#fff;{FONT}font-size:14px;color:#222;line-height:1.6;">

      <p style="margin:0 0 16px;">Dear team,</p>

      <p style="margin:0 0 12px;">
        Please find the <strong>{file_xlsx}</strong> details as follow:
      </p>

      <ol style="margin:0 0 20px;padding-left:22px;">

        <li style="margin-bottom:12px;">
          <strong>Cycle:</strong> Three - Five week.
        </li>

        <li style="margin-bottom:4px;">
          <strong>Swap:</strong> within 30 hours since schedule email sent out.
          <ul style="margin:8px 0 0;padding-left:20px;list-style:disc;color:#333;">
            <li style="margin-bottom:4px;">
              Swap shift <em>(day &amp; night)</em>: Seek approval from
              <strong>Mr. Puneet</strong> via email.
            </li>
            <li style="margin-bottom:4px;">Only swap between 1 LOB.</li>
            <li>
              Shift Swap form:&nbsp;
              <a href="{wo_swap_link}" style="color:#1155CC;font-weight:bold;">
                Click here to submit
              </a>
            </li>
          </ul>
        </li>

      </ol>

      <hr style="border:none;border-top:1px solid #e0e0e0;margin:20px 0;">

      <table style="border-collapse:collapse;margin:0 0 22px;">
        <tr>
          <td style="padding:5px 16px 5px 0;color:#555;font-size:13px;white-space:nowrap;">
            &#128197;&nbsp;Week
          </td>
          <td style="padding:5px 0;font-size:13px;color:#222;">
            <strong>{monday_lbl} &rarr; {sunday_lbl}</strong>
          </td>
        </tr>
        <tr>
          <td style="padding:5px 16px 5px 0;color:#555;font-size:13px;white-space:nowrap;">
            &#128196;&nbsp;File
          </td>
          <td style="padding:5px 0;font-size:13px;font-weight:bold;color:#222;">
            {file_xlsx}
          </td>
        </tr>
      </table>

      <p style="margin:0 0 28px;">
        <a href="{folder_link}"
           style="display:inline-block;padding:10px 22px;
                  background:#0078D4;color:#fff;border-radius:5px;
                  text-decoration:none;font-size:14px;font-weight:bold;{FONT}">
          &#128194;&nbsp;Open Schedule Folder on SharePoint
        </a>
      </p>

      <p style="font-size:13px;color:#444;margin:0;line-height:1.8;
                border-top:1px solid #e0e0e0;padding-top:16px;">
        Thanks &amp; Regards,<br>
        <strong>Chinh Nguyen</strong><br>
        Analyst, WFM Real Time Management<br>
        Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street,<br>
        Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>
        Ph: +84 986 473 419&nbsp;&nbsp;|&nbsp;&nbsp;
        <a href="mailto:huuchinh.nguyen@concentrix.com" style="color:#1155CC;">
          huuchinh.nguyen@concentrix.com
        </a>
      </p>

    </div>
    """

# ── Outlook sender ───────────────────────────────────────────
def send_outlook_email(to: str, cc: str, subject: str, body: str) -> None:
    """Send HTML email via Outlook desktop (win32com)."""
    pythoncom.CoInitialize()
    was_running = any(
        p.name().lower() == "outlook.exe"
        for p in psutil.process_iter(["name"])
    )
    if not was_running:
        for exe in [
            r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
            r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
        ]:
            if os.path.exists(exe):
                subprocess.Popen([exe])
                print("[Outlook] Starting...")
                break
        for _ in range(30):
            time.sleep(1)
            try: win32com.client.GetActiveObject("Outlook.Application"); break
            except Exception: pass
    try:
        ol = win32com.client.Dispatch("Outlook.Application")
        ol.GetNamespace("MAPI").Logon()
        mail          = ol.CreateItem(0)
        mail.To       = to
        mail.CC       = cc
        mail.Subject  = subject
        mail.HTMLBody = body
        mail.Send()
        print(f"[Outlook] Email sent.")
        print(f"  To      : {to}")
        print(f"  Subject : {subject}")
        time.sleep(3)
    finally:
        if not was_running:
            try: ol.Quit(); print("[Outlook] Closed.")
            except Exception: pass

# ── Main ─────────────────────────────────────────────────────
print("\n" + "="*60)

latest_file = get_latest_schedule_file(SCHEDULE_OPS_FOLDER)

week_str = WEEK.strip()
if not week_str:
    week_str = auto_detect_week(latest_file, SCHEDULE_OPS_FOLDER)
else:
    print(f"[Week] Manual: {week_str}")

monday_lbl, sunday_lbl = parse_week_label(week_str)
print(f"[Week] Range : {monday_lbl} -> {sunday_lbl}")

subject = f"[Draft] {latest_file.stem}" 
body = build_html_body(
    file_stem    = latest_file.stem,
    folder_link  = FOLDER_LINK.strip(),
    wo_swap_link = WO_SWAP_LINK.strip(),
    monday_lbl   = monday_lbl,
    sunday_lbl   = sunday_lbl,
)

# Always preview first
print("\n" + "="*60)
print("[Preview] Email preview:")
display(HTML(body))

print("="*60)
print(f"  Subject : {subject}")
print(f"  To      : {EMAIL_TO}")
print(f"  Folder  : {FOLDER_LINK[:80]}...")
print("="*60)

if not SEND_EMAIL:
    print("[SKIP] SEND_EMAIL = False — preview only.")
else:
    _ans = input("\nSend DRAFT email? (yes/no): ").strip().lower()
    print("="*60)
    if _ans == "yes":
        send_outlook_email(EMAIL_TO, EMAIL_CC, subject, body)
        print("\n[DONE] Draft email sent.")

        # Save EntryID to cache so Cell 3 can Reply-All to this email
        try:
            import json as _json
            pythoncom.CoInitialize()
            _ol  = win32com.client.Dispatch("Outlook.Application")
            _ns  = _ol.GetNamespace("MAPI")
            _sent = _ns.GetDefaultFolder(5)   # Sent Items
            _items = _sent.Items
            _items.Sort("[SentOn]", True)
            _wb   = re.search(r"WB\d{4}", latest_file.name).group(0)
            _cf   = Path(f"{HOME}/.schedule_draft_cache.json")
            _cache = {}
            try: _cache = _json.loads(_cf.read_text())
            except: pass
            for _i, _m in enumerate(_items):
                if _i >= 30: break
                try:
                    if _m.Subject == subject:
                        _cache[_wb] = {"entry_id": _m.EntryID,
                                       "store_id": _m.Parent.StoreID,
                                       "subject":  subject}
                        _cf.write_text(_json.dumps(_cache, indent=2))
                        print(f"[Cache] EntryID saved for {_wb} → ready for Final (Cell 3)")
                        break
                except: pass
        except Exception as _e:
            print(f"[Cache] WARNING: could not save EntryID: {_e}")
    else:
        print("[CANCELLED] Email not sent.")

# ── Teams notification (Draft) ───────────────────────────────────
def _teams_draft_html():
    return (
        '<div style="font-family:Calibri,Arial,sans-serif;font-size:13px;">'
        f'<p style="margin:0 0 6px;">Dear team,</p>'
        f'<p style="margin:0 0 6px;">Please find the <strong>[DRAFT] {latest_file.stem}.xlsx</strong> details as follow:</p>'
        '<ol style="margin:0 0 8px;padding-left:20px;">'
        '<li style="margin-bottom:4px;"><strong>Cycle:</strong> Three - Five week.</li>'
        '<li style="margin-bottom:4px;"><strong>Swap:</strong> within 30 hours since schedule email sent out.'
        '<ul style="margin:4px 0;padding-left:16px;">'
        '<li>Swap shift <em>(day &amp; night)</em>: Seek approval from <strong>Mr. Puneet</strong> via email.</li>'
        '<li>Only swap between 1 LOB.</li>'
        '</ul></li>'
        '</ol>'
        f'<p style="margin:0 0 6px;">📅 <strong>Week:</strong> {monday_lbl} &rarr; {sunday_lbl}<br>'
        f'📄 <strong>File:</strong> {latest_file.stem}.xlsx</p>'
        f'<p style="margin:0;">'
        f'<a href="{FOLDER_LINK}" style="color:#0078D4;">📂 Open Schedule Folder on SharePoint</a><br>'
        f'<a href="{WO_SWAP_LINK}" style="color:#0078D4;">📝 Shift Swap form</a>'
        '</p></div>'
    )

_t_ans = input("\nSend DRAFT to Teams? (yes/no): ").strip().lower()
if _t_ans == "yes":
    import requests as _req, json as _json
    _r = _req.post(
        "https://default599e51d62f8c43478e591f795a51a9.8c.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/30e46f2733bc4e48a92dc32f90ba9329/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=xkD_H8_VvQh_XzhybXDxV3_gWFyC0E4-3Bpe_MJDJ44",
        headers={"Content-Type": "application/json"},
        data=_json.dumps({"html": _teams_draft_html()}),
        timeout=30
    )
    _ok = _r.status_code in (200, 202)
    print(f"[Teams] Draft {'OK' if _ok else 'FAIL'} — HTTP {_r.status_code}"
          + ("" if _ok else f" | {_r.text[:120]}"))
else:
    print("[Teams] Skipped.")


[Config]
  Schedule folder : OK
  Folder link     : https://cnxmail.sharepoint.com/:f:/r/sites/WFM-Expedia-HCM/Branding%20...
  WO Swap link    : https://cnxmail.sharepoint.com/:x:/s/WFM-Expedia-HCM/IQAGptitR5FiTZNPE...


[File] Expedia VN_Schedule_WB0831.xlsx
       C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Schedule\Schedule (Ops version)\2026\03-Mar\Expedia VN_Schedule_WB0831.xlsx
[Week] Manual: 2026_08_31
[Week] Range : 31/08/2026 -> 06/09/2026

[Preview] Email preview:


📅 Week,31/08/2026 → 06/09/2026
📄 File,Expedia VN_Schedule_WB0831.xlsx


  Subject : [Draft] Expedia VN_Schedule_WB0831
  To      : van.tran@concentrix.com;puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;myduyen.ly@concentrix.com;mylinh.do@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com;ExpediaVN_Training_Team@concentrix.com;ExpediaVN_QA_Team@concentrix.com;
  Folder  : https://cnxmail.sharepoint.com/:f:/r/sites/WFM-Expedia-HCM/Branding%20files/Sche...
[CANCELLED] Email not sent.
[Teams] Skipped.


In [12]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — FINAL EMAIL  (Reply-All to Draft)
# Run AFTER Cell 2 draft has been sent.
# ═══════════════════════════════════════════════════════════════
import pythoncom, win32com.client, time, os, subprocess, psutil
from IPython.display import display, HTML

import os, traceback

# Defensive: redefine HOME in case Cell 2 was not run in this session
HOME = os.path.expanduser("~").replace("\\", "/")

# Ensure Cell 1 + 2 variables exist
try:
    _ = latest_file, monday_lbl, sunday_lbl, FOLDER_LINK
except NameError:
    raise RuntimeError("Run Cell 1 then Cell 2 (send draft) before running Cell 3.")

DRAFT_SUBJECT = f"[Draft] {latest_file.stem}"
FINAL_SUBJECT = f"[Final] {latest_file.stem}"

print(f"[Final] Looking for draft : '{DRAFT_SUBJECT}'")
print(f"[Final] Reply subject     : '{FINAL_SUBJECT}'")

# ── RTA Schedule folder ──────────────────────────────────────
RTA_FOLDER = (
    f"{HOME}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
    f"/Schedule/Schedule (RTA version)/2026"
)

# ── Read step_2 sheet → styled HTML table ────────────────────
def read_step2_html(ops_file: Path, rta_folder: str) -> str:
    import pandas as pd

    # Match WB number from ops filename (e.g. WB0831)
    wb_match = re.search(r"WB(\d{4})", ops_file.name)
    if not wb_match:
        raise ValueError(f"Cannot extract WB number from: {ops_file.name}")
    wb_num = wb_match.group(1)

    # Find matching RTA file
    rta_candidates = sorted(
        [f for f in Path(rta_folder).rglob(f"*WB{wb_num}*.xlsx")
         if not f.name.startswith("~$")]
    )
    if not rta_candidates:
        raise FileNotFoundError(f"No RTA file for WB{wb_num} in:\n  {rta_folder}")
    rta_file = rta_candidates[0]
    print(f"[Step_2] RTA file : {rta_file.name}")

    # List sheets + find step_2 (case-insensitive)
    xl     = pd.ExcelFile(str(rta_file))
    sheets = xl.sheet_names
    print(f"[Step_2] Sheets   : {sheets}")
    sheet  = next((s for s in sheets if s.lower() == "step_2"), None)
    if sheet is None:
        xl.close()
        raise ValueError(f"Sheet 'step_2' not found. Available: {sheets}")

    df_raw = pd.read_excel(xl, sheet_name=sheet, dtype=str, header=0).fillna("")
    xl.close()
    print(f"[Step_2] Raw cols ({len(df_raw.columns)}): {list(df_raw.columns)}")

    # ── Keep only the 18 relevant columns ───────────────────────
    DAYS     = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    EXPECTED = (["LOB", "Requester", "IEX", "Agent Name"]
                + [f"Old {d}" for d in DAYS]
                + [f"New {d}" for d in DAYS])

    # Match existing columns (case-insensitive, strip whitespace)
    col_map = {str(c).strip(): str(c) for c in df_raw.columns}
    keep    = [col_map[e] for e in EXPECTED if e in col_map]
    missing = [e for e in EXPECTED if e not in col_map]
    if missing:
        print(f"[Step_2] WARNING — cols not found: {missing}")
    df = df_raw[keep].copy()

    # Drop rows where ALL cells are blank (empty rows in Excel between header and data)
    df = df[df.apply(lambda r: r.str.strip().any(), axis=1)].reset_index(drop=True)
    print(f"[Step_2] Final: {len(df)} rows × {len(df.columns)} cols")

    # ── Styled HTML table ────────────────────────────────────
    # Key Outlook fix: wrap each cell's text in <p style="margin:0;padding:0">
    # This suppresses the default paragraph spacing Outlook Word engine adds.
    P_BASE = ("margin:0;padding:0;"
              "font-family:Calibri,Arial,sans-serif;font-size:11px;"
              "line-height:13px;mso-line-height-rule:exactly;")
    TH_S   = ("background:#7B3F00;color:#fff;"
               "padding:2px 6px;border:1px solid #555;"
               "text-align:center;white-space:nowrap;")
    TD_DEF  = "padding:0 6px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#fff;"
    TD_OFF  = "padding:0 6px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#BDBDBD;"
    TD_TERM = "padding:0 6px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#424242;"

    def make_td(val):
        v  = str(val).strip()
        vu = v.upper()
        if vu == "OFF":
            return (f'<td style="{TD_OFF}">'
                    f'<p style="{P_BASE}color:#333;">{v}</p></td>')
        if "TERMIN" in vu:
            return (f'<td style="{TD_TERM}">'
                    f'<p style="{P_BASE}color:#fff;">{v}</p></td>')
        return (f'<td style="{TD_DEF}">'
                f'<p style="{P_BASE}color:#000;">{v}</p></td>')

    # ── Date-aware column headers ────────────────────────────
    from datetime import datetime as _dt, timedelta as _td
    DAYS_ABR = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    COL_DATE_MAP = {}
    wb_date = re.search(r"WB(\d{2})(\d{2})", rta_file.name)
    if wb_date:
        yr      = int(Path(rta_folder.rstrip("/")).name)
        new_mon = _dt(yr, int(wb_date.group(1)), int(wb_date.group(2)))
        old_mon = new_mon - _td(days=7)
        for i, d in enumerate(DAYS_ABR):
            old_dt = (old_mon + _td(days=i)).strftime("%d/%m")
            new_dt = (new_mon + _td(days=i)).strftime("%d/%m")
            # Display: only date (no Old/New text), colour distinguishes them
            COL_DATE_MAP[f"Old {d}"] = (f"{d}<br>{old_dt}", "#B45309")  # amber = old
            COL_DATE_MAP[f"New {d}"] = (f"{d}<br>{new_dt}", "#1565C0")  # blue  = new

    BASE_TH_BG = "#7B3F00"  # dark brown for base cols

    def make_th(col):
        display_text, bg = COL_DATE_MAP.get(col, (col, BASE_TH_BG))
        th_s = (f"background:{bg};color:#fff;"
                "padding:2px 6px;border:1px solid #555;"
                "text-align:center;white-space:nowrap;")
        return (f'<th style="{th_s}">'
                f'<p style="{P_BASE}color:#fff;font-weight:bold;text-align:center;">' 
                f'{display_text}</p></th>')

    header = "<tr>" + "".join(make_th(c) for c in df.columns) + "</tr>"
    rows   = "".join(
        "<tr>" + "".join(make_td(v) for v in row) + "</tr>"
        for _, row in df.iterrows()
    )
    tbl_style = ("border-collapse:collapse;margin-bottom:16px;"
                 "mso-table-lspace:0pt;mso-table-rspace:0pt;")
    return (
        '<p style="font-weight:bold;font-family:Calibri,Arial,sans-serif;margin:16px 0 4px;">'
        'Schedule Detail (Step 2):</p>'
        '<div style="overflow-x:auto;">'
        f'<table cellpadding="0" cellspacing="0" style="{tbl_style}">{header}{rows}</table>'
        '</div>'
    )

# ── Build simplified final body (no Cycle/Swap section) ──────
def build_final_body(file_stem, folder_link, monday_lbl, sunday_lbl, step2_html=""):
    FONT = "font-family:Calibri,Arial,sans-serif;"
    file_xlsx = f"{file_stem}.xlsx"
    TD_L = "padding:2px 14px 2px 0;color:#555;font-size:13px;white-space:nowrap;"
    TD_R_W = "padding:2px 0;font-size:13px;color:#222;"
    TD_R_F = "padding:2px 0;font-size:13px;font-weight:bold;color:#222;"
    return f"""
    <!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>
    <o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->
    <div style="padding:24px 28px;background:#fff;{FONT}font-size:14px;color:#222;line-height:1.4;">

      <p style="margin:0 0 8px;">Dear team,</p>

      <p style="margin:0 0 8px;">
        Please find the <strong>FINAL {file_xlsx}</strong> details as follow:
      </p>

      <table style="border-collapse:collapse;margin:0 0 6px;">
        <tr>
          <td style="{TD_L}">&#128197;&nbsp;Week</td>
          <td style="{TD_R_W}"><strong>{monday_lbl} &rarr; {sunday_lbl}</strong></td>
        </tr>
        <tr>
          <td style="{TD_L}">&#128196;&nbsp;File</td>
          <td style="{TD_R_F}">{file_xlsx}</td>
        </tr>
      </table>

      <p style="margin:0 0 16px;">
        <a href="{folder_link}"
           style="display:inline-block;padding:8px 20px;
                  background:#0078D4;color:#fff;border-radius:5px;
                  text-decoration:none;font-size:13px;font-weight:bold;{FONT}">
          &#128194;&nbsp;Open Schedule Folder on SharePoint
        </a>
      </p>

      {step2_html}

      <p style="font-size:13px;color:#444;margin:0;line-height:1.8;
                border-top:1px solid #e0e0e0;padding-top:16px;">
        Thanks &amp; Regards,<br>
        <strong>Chinh Nguyen</strong><br>
        Analyst, WFM Real Time Management<br>
        Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street,<br>
        Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>
        Ph: +84 986 473 419&nbsp;&nbsp;|&nbsp;&nbsp;
        <a href="mailto:huuchinh.nguyen@concentrix.com" style="color:#1155CC;">
          huuchinh.nguyen@concentrix.com
        </a>
      </p>

    </div>
    """

# ── Find draft in Sent Items ──────────────────────────────────
def find_draft_in_sent(draft_subj):
    """
    Search Sent Items for the draft email.
    Uses case-insensitive substring match on the WB number for reliability.
    """
    pythoncom.CoInitialize()
    ol = win32com.client.Dispatch("Outlook.Application")
    ns = ol.GetNamespace("MAPI")
    ns.Logon()

    sent = ns.GetDefaultFolder(5)   # 5 = olFolderSentMail
    sent.Items.Sort("[SentOn]", True)   # newest first

    # Extract WB key from expected subject for flexible matching
    import re as _re
    wb_key = _re.search(r"WB\d{4}", draft_subj)
    wb_key = wb_key.group(0) if wb_key else draft_subj

    print(f"[Search] Looking in Sent Items for: '{draft_subj}'")

    candidates = []
    checked    = 0
    for item in sent.Items:
        try:
            subj = getattr(item, "Subject", "") or ""
            checked += 1
            if checked <= 5:
                print(f"  [Sent #{checked}] {subj!r}")
            # Must contain [Draft] AND the WB key — avoids matching old emails
            if ("[draft]" in subj.lower()
                    and wb_key.lower() in subj.lower()
                    and "final" not in subj.lower()):
                candidates.append(item)
                break   # take the most recent match only
        except Exception:
            continue

    if not candidates:
        print(f"[Info] No [Draft] email found for {wb_key} — will send as new email.")
        raise ValueError("draft_not_found")

    match = candidates[0]
    print(f"[Draft found] Subject : {match.Subject}")
    print(f"              Sent at : {match.SentOn}")
    return match, ol

# ── Send Reply-All ────────────────────────────────────────────
def send_final_reply():
    import re as _re

    was_on = any(p.name().lower() == "outlook.exe" for p in psutil.process_iter(["name"]))
    if not was_on:
        for exe in [
            r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
            r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
        ]:
            if os.path.exists(exe):
                subprocess.Popen([exe])
                print("[Outlook] Starting...")
                break
        for _ in range(30):
            time.sleep(1)
            try: win32com.client.GetActiveObject("Outlook.Application"); break
            except Exception: pass

    pythoncom.CoInitialize()
    ol = win32com.client.Dispatch("Outlook.Application")
    ol.GetNamespace("MAPI").Logon()

    # ── Try to find draft → Reply-All; fallback → new email ──
    draft_item = None
    try:
        draft_item, _ = find_draft_in_sent(DRAFT_SUBJECT)
    except ValueError as _e:
        print(f"[Info] No draft found ({_e.__class__.__name__}) — sending as new email.")

    try:
        if draft_item is not None:
            # ── Reply-All to draft ────────────────────────────
            reply = draft_item.ReplyAll()
            reply.Subject = FINAL_SUBJECT

            raw_html = reply.HTMLBody
            quote_match = _re.search(r'<div\s+id=["\']divRplyFwdMsg["\']',
                                     raw_html, _re.IGNORECASE)
            if quote_match:
                quoted_thread = raw_html[quote_match.start():]
            else:
                hr_match = _re.search(
                    r'<hr\s[^>]*style=["\'][^"\']*(?:solid|2pt)[^"\']*["\'][^>]*>',
                    raw_html, _re.IGNORECASE)
                quoted_thread = raw_html[hr_match.start():] if hr_match else ""

            reply.HTMLBody = final_body + (f'<div>{quoted_thread}</div>' if quoted_thread else "")
            reply.Send()
            print(f"[Outlook] Final sent as Reply-All to draft.")
            print(f"  Subject : {FINAL_SUBJECT}")

        else:
            # ── New standalone email ──────────────────────────
            mail          = ol.CreateItem(0)
            mail.To       = EMAIL_TO
            mail.CC       = EMAIL_CC
            mail.Subject  = FINAL_SUBJECT
            mail.HTMLBody = final_body
            mail.Send()
            print(f"[Outlook] Final sent as new email (no draft found).")
            print(f"  To      : {EMAIL_TO}")
            print(f"  Subject : {FINAL_SUBJECT}")

        time.sleep(2)
    finally:
        if not was_on:
            try: ol.Quit(); print("[Outlook] Closed.")
            except Exception: pass

# ── Preview ───────────────────────────────────────────────────
print(f"\n[Step_2] RTA folder : {RTA_FOLDER}")
print(f"[Step_2] Folder OK  : {Path(RTA_FOLDER).exists()}")
try:
    step2_html = read_step2_html(latest_file, RTA_FOLDER)
    print("[Step_2] ✅ Table built successfully.")
except Exception as _e:
    import traceback
    print(f"\n[Step_2] ❌ {type(_e).__name__}: {_e}")
    traceback.print_exc()
    step2_html = f'<p style="color:#c00;font-family:Calibri;font-size:12px;">'                 f'[Step_2 error: {type(_e).__name__}: {_e}]</p>'

final_body = build_final_body(
    file_stem  = latest_file.stem,
    folder_link= FOLDER_LINK.strip(),
    monday_lbl = monday_lbl,
    sunday_lbl = sunday_lbl,
    step2_html = step2_html,
)

print("\n" + "="*60)
print("[Preview] Final email body:")
display(HTML(final_body))

print("="*60)
print(f"  Draft    : {DRAFT_SUBJECT}")
print(f"  Final    : {FINAL_SUBJECT}")
print(f"  Action   : Reply-All to draft in Sent Items")
print("="*60)

if not SEND_EMAIL:
    print("[SKIP] SEND_EMAIL = False — preview only.")
else:
    _ans = input("\nSend FINAL email (Reply-All)? (yes/no): ").strip().lower()
    print("="*60)
    if _ans == "yes":
        send_final_reply()
        print("\n[DONE] Final email sent successfully.")
    else:
        print("[CANCELLED] Final email not sent.")




# ── Teams notification (Final) ───────────────────────────────────
def _teams_final_html():
    return (
        '<div style="font-family:Calibri,Arial,sans-serif;font-size:13px;">'
        f'<p style="margin:0 0 6px;">Dear team,</p>'
        f'<p style="margin:0 0 6px;">Please find the <strong>[FINAL] {latest_file.stem}.xlsx</strong> details as follow:</p>'
        f'<p style="margin:0 0 6px;">📅 <strong>Week:</strong> {monday_lbl} &rarr; {sunday_lbl}<br>'
        f'📄 <strong>File:</strong> {latest_file.stem}.xlsx</p>'
        f'<p style="margin:0;">'
        f'<a href="{FOLDER_LINK}" style="color:#0078D4;">📂 Open Schedule Folder on SharePoint</a>'
        '</p></div>'
    )

_t_ans = input("\nSend FINAL to Teams? (yes/no): ").strip().lower()
if _t_ans == "yes":
    import requests as _req, json as _json
    _r = _req.post(
        "https://default599e51d62f8c43478e591f795a51a9.8c.environment.api.powerplatform.com:443/powerautomate/automations/direct/workflows/b7d26f3e06b04f62b5bf0eed791caaf5/triggers/manual/paths/invoke?api-version=1&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=MJH_18i1A8L-n6BFpdVL4xzP_U-3izTgle76wcqWcZE",
        headers={"Content-Type": "application/json"},
        data=_json.dumps({"html": _teams_final_html()}),
        timeout=30
    )
    _ok = _r.status_code in (200, 202)
    print(f"[Teams] Final {'OK' if _ok else 'FAIL'} — HTTP {_r.status_code}"
          + ("" if _ok else f" | {_r.text[:120]}"))
else:
    print("[Teams] Skipped.")


[Final] Looking for draft : '[Draft] Expedia VN_Schedule_WB0831'
[Final] Reply subject     : '[Final] Expedia VN_Schedule_WB0831'

[Step_2] RTA folder : C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (RTA version)/2026
[Step_2] Folder OK  : True
[Step_2] RTA file : Schedule_WB0831.xlsx
[Step_2] Sheets   : ['Summary', 'Planned', 'Swap', 'Schedule_FINAL', 'Schedule_ANALYSIS', 'Schedule_VN', 'Schedule_Team', 'Schedule_IEX', 'Schedule_1', 'Schedule_2', 'HO', 'Step_2', 'HC_Extend', 'EWS', '2 .mismatch_iex_final', '1. mismatch_iex', 'Mini_Team']
[Step_2] Raw cols (42): ['no', 'Requester', 'IEX', 'Agent Name', 'Old Mon', 'Old Tue', 'Old Wed', 'Old Thu', 'Old Fri', 'Old Sat', 'Old Sun', 'New Mon', 'New Tue', 'New Wed', 'New Thu', 'New Fri', 'New Sat', 'New Sun', 'Unnamed: 18', 'Unnamed: 19', 'Historical_Edit', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Schedule_2', 'Unnamed: 29', 

  Draft    : [Draft] Expedia VN_Schedule_WB0831
  Final    : [Final] Expedia VN_Schedule_WB0831
  Action   : Reply-All to draft in Sent Items
[CANCELLED] Final email not sent.
[Teams] Final OK — HTTP 202
